# Random Forest LAI Baseline: Clean Leakage-Free Split

This notebook retrains the Random Forest LAI baseline using only the leakage-free train, validation, and test CSV files. The nine predictors are recomputed directly from each raw RGB image. The validation split is used only as a diagnostic; the finalized model uses the clean training split and is evaluated once on the clean test split.

In [17]:
from pathlib import Path
import hashlib
import json
import math
import warnings

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
np.random.seed(42)

project_root = Path.cwd().resolve()
if not (project_root / 'data' / 'lai_train_clean.csv').exists():
    for candidate in [project_root.parent, *project_root.parents]:
        if (candidate / 'data' / 'lai_train_clean.csv').exists():
            project_root = candidate
            break

DATA_DIR = project_root / 'data'
RESULTS_DIR = project_root / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

CLEAN_FILES = {
    'train': 'lai_train_clean.csv',
    'validation': 'lai_val_clean.csv',
    'test': 'lai_test_clean.csv',
}
FEATURE_NAMES = [
    'mean_red', 'mean_green', 'mean_blue',
    'std_red', 'std_green', 'std_blue',
    'mean_brightness', 'green_to_red_ratio', 'green_to_blue_ratio',
]

print(f'Project root: {project_root}')
for split_name, filename in CLEAN_FILES.items():
    print(f'{split_name.title()} CSV: {(DATA_DIR / filename).exists()}')

Project root: C:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water
Train CSV: True
Validation CSV: True
Test CSV: True


In [18]:
def load_clean_split(filename):
    return pd.read_csv(DATA_DIR / filename)

train_df = load_clean_split(CLEAN_FILES['train'])
val_df = load_clean_split(CLEAN_FILES['validation'])
test_df = load_clean_split(CLEAN_FILES['test'])

required_columns = {'image_filename', 'image_path', 'LAI'}
for split_name, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise ValueError(f'{split_name} is missing required columns: {sorted(missing_columns)}')

def resolve_image_path(path_value):
    path = Path(str(path_value))
    if path.is_absolute():
        return path
    candidates = [DATA_DIR / path, project_root / path, project_root.parent / path]
    for candidate in candidates:
        if candidate.exists():
            return candidate.resolve()
    return (project_root / path).resolve()

for frame in [train_df, val_df, test_df]:
    frame['resolved_image_path'] = frame['image_path'].map(resolve_image_path)

all_frames = {'train': train_df, 'validation': val_df, 'test': test_df}
missing_images = [(split_name, str(path)) for split_name, frame in all_frames.items() for path in frame['resolved_image_path'] if not path.is_file()]
print(f'Train samples: {len(train_df)}')
print(f'Validation samples: {len(val_df)}')
print(f'Test samples: {len(test_df)}')
print(f'Missing raw RGB images: {len(missing_images)}')
if missing_images:
    raise FileNotFoundError(f'Missing raw RGB images: {missing_images[:5]}')

Train samples: 234
Validation samples: 51
Test samples: 51
Missing raw RGB images: 0


In [19]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as file_handle:
        for chunk in iter(lambda: file_handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

identity_rows = []
for split_name, frame in all_frames.items():
    for _, row in frame.iterrows():
        identity_rows.append({
            'split': split_name,
            'image': str(row['image_filename']),
            'resolved_path': str(row['resolved_image_path']),
            'sha256': sha256_file(row['resolved_image_path']),
        })
identity_df = pd.DataFrame(identity_rows)

hash_split_counts = identity_df.groupby('sha256')['split'].nunique()
cross_split_hashes = int((hash_split_counts > 1).sum())
path_split_counts = identity_df.groupby('resolved_path')['split'].nunique()
cross_split_paths = int((path_split_counts > 1).sum())

print(f'Cross-split identical image hashes: {cross_split_hashes}')
print(f'Cross-split duplicate paths: {cross_split_paths}')
if cross_split_hashes != 0 or cross_split_paths != 0:
    raise ValueError('The clean split has cross-split image overlap; training stopped.')

Cross-split identical image hashes: 0
Cross-split duplicate paths: 0


In [20]:
def compute_rgb_features(image_path):
    rgb = np.asarray(Image.open(image_path).convert('RGB'), dtype=np.float32)
    red = rgb[:, :, 0]
    green = rgb[:, :, 1]
    blue = rgb[:, :, 2]
    brightness = (red + green + blue) / 3.0
    feature_values = {
        'mean_red': red.mean(),
        'mean_green': green.mean(),
        'mean_blue': blue.mean(),
        'std_red': red.std(),
        'std_green': green.std(),
        'std_blue': blue.std(),
        'mean_brightness': brightness.mean(),
        'green_to_red_ratio': green.mean() / (red.mean() + 1e-8),
        'green_to_blue_ratio': green.mean() / (blue.mean() + 1e-8),
    }
    return [feature_values[name] for name in FEATURE_NAMES]

def build_feature_matrix(frame):
    return np.asarray([compute_rgb_features(path) for path in frame['resolved_image_path']], dtype=np.float32)

X_train = build_feature_matrix(train_df)
X_val = build_feature_matrix(val_df)
X_test = build_feature_matrix(test_df)
y_train = train_df['LAI'].to_numpy(dtype=float)
y_val = val_df['LAI'].to_numpy(dtype=float)
y_test = test_df['LAI'].to_numpy(dtype=float)

print(f'Features: {len(FEATURE_NAMES)}')
print('Feature names:', ', '.join(FEATURE_NAMES))
print(f'Feature matrix shapes: train={X_train.shape}, validation={X_val.shape}, test={X_test.shape}')

Features: 9
Feature names: mean_red, mean_green, mean_blue, std_red, std_green, std_blue, mean_brightness, green_to_red_ratio, green_to_blue_ratio
Feature matrix shapes: train=(234, 9), validation=(51, 9), test=(51, 9)


In [21]:
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train, y_train)

val_pred = rf_model.predict(X_val)
val_metrics = {
    'MAE': mean_absolute_error(y_val, val_pred),
    'RMSE': math.sqrt(mean_squared_error(y_val, val_pred)),
    'R2': r2_score(y_val, val_pred),
}
print('Validation diagnostic (not used to tune):')
for metric_name, metric_value in val_metrics.items():
    print(f'{metric_name}: {metric_value:.9f}')

test_pred = rf_model.predict(X_test)
mae = mean_absolute_error(y_test, test_pred)
rmse = math.sqrt(mean_squared_error(y_test, test_pred))
r2 = r2_score(y_test, test_pred)

print('\nRANDOM FOREST â€” CLEAN LEAKAGE-FREE SPLIT')
print('-----------------------------------------')
print(f'Train samples: {len(train_df)}')
print(f'Validation samples: {len(val_df)}')
print(f'Test samples: {len(test_df)}')
print(f'Features: {len(FEATURE_NAMES)}')
print()
print(f'MAE: {mae:.9f}')
print(f'RMSE: {rmse:.9f}')
print(f'RÂ²: {r2:.9f}')

Validation diagnostic (not used to tune):
MAE: 0.003934226
RMSE: 0.006222206
R2: 0.950456790

RANDOM FOREST â€” CLEAN LEAKAGE-FREE SPLIT
-----------------------------------------
Train samples: 234
Validation samples: 51
Test samples: 51
Features: 9

MAE: 0.006192920
RMSE: 0.009753515
RÂ²: 0.865564843


In [22]:
import joblib
joblib.dump(rf_model, '../../DatasetLAI/lai_rf_model.joblib')

['../../DatasetLAI/lai_rf_model.joblib']

In [23]:
metrics = {
    'MAE': float(mae),
    'RMSE': float(rmse),
    'R2': float(r2),
    'features': FEATURE_NAMES,
    'train_samples': int(len(train_df)),
    'validation_samples': int(len(val_df)),
    'test_samples': int(len(test_df)),
    'cross_split_identical_image_hashes': cross_split_hashes,
    'cross_split_duplicate_paths': cross_split_paths,
    'random_seed': 42,
    'model': 'RandomForestRegressor(default settings)',
}
(RESULTS_DIR / 'lai_rf_clean_metrics.json').write_text(json.dumps(metrics, indent=2))

predictions_df = pd.DataFrame({
    'image': test_df['image_filename'].astype(str),
    'actual_LAI': y_test,
    'predicted_LAI': test_pred,
    'error': test_pred - y_test,
})
predictions_df.to_csv(RESULTS_DIR / 'lai_rf_clean_predictions.csv', index=False)

plt.figure(figsize=(6, 6))
plt.scatter(y_test, test_pred, alpha=0.7, s=40, color='#1f77b4')
plot_min = min(y_test.min(), test_pred.min())
plot_max = max(y_test.max(), test_pred.max())
padding = max((plot_max - plot_min) * 0.05, 0.001)
plt.plot([plot_min - padding, plot_max + padding], [plot_min - padding, plot_max + padding], color='black', linestyle='--', linewidth=1.5, label='Ideal fit')
plt.xlabel('Actual LAI')
plt.ylabel('Predicted LAI')
plt.title('Actual LAI vs Predicted LAI (Clean RF)')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lai_rf_clean_actual_vs_predicted.png', dpi=200)
plt.close()

plt.figure(figsize=(6, 4.5))
plt.hist(test_pred - y_test, bins=20, color='#2ca02c', edgecolor='black')
plt.axvline(0, color='black', linestyle='--', linewidth=1.5)
plt.xlabel('Prediction Error (Predicted - Actual)')
plt.ylabel('Count')
plt.title('Prediction Error Distribution (Clean RF)')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'lai_rf_clean_error_distribution.png', dpi=200)
plt.close()

print('Created files:')
for artifact in ['lai_rf_clean_metrics.json', 'lai_rf_clean_predictions.csv', 'lai_rf_clean_actual_vs_predicted.png', 'lai_rf_clean_error_distribution.png']:
    print(f'- {RESULTS_DIR / artifact}')

Created files:
- C:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water\results\lai_rf_clean_metrics.json
- C:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water\results\lai_rf_clean_predictions.csv
- C:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water\results\lai_rf_clean_actual_vs_predicted.png
- C:\Users\nihar\OneDrive\Desktop\Water To Pixels\pixels-to-water\results\lai_rf_clean_error_distribution.png


In [24]:
old_metrics_path = RESULTS_DIR / 'lai_baseline_metrics.json'
old_metrics = json.loads(old_metrics_path.read_text())
print('Difference between old and clean RF metrics (clean - old):')
for metric_name in ['MAE', 'RMSE', 'R2']:
    print(f'{metric_name}: {metrics[metric_name] - old_metrics[metric_name]:+.9f}')

print('\nThe Random Forest LAI baseline was retrained using the leakage-free dataset split. These results replace the previous RF test results for model comparison.')
print('\nSuggested Git commit message:')
print('Re-evaluate Random Forest LAI on leakage-free split')

Difference between old and clean RF metrics (clean - old):
MAE: +0.001127790
RMSE: +0.001927161
R2: -0.046659271

The Random Forest LAI baseline was retrained using the leakage-free dataset split. These results replace the previous RF test results for model comparison.

Suggested Git commit message:
Re-evaluate Random Forest LAI on leakage-free split


## Conclusion

The Random Forest LAI baseline was retrained using the leakage-free dataset split. These results replace the previous RF test results for model comparison.